# Assignment 5: Exploratory Data Analysis & Pattern Mining (Satellite Dataset)

**Course Unit:** Unit VI – Exploratory Data Analysis Tools  
**Student Name:** [SOURAV SUMAN]  

---

## Overview

This notebook implements EDA, clustering, and classification on the Satellite dataset (6,435 instances, 36 spectral features).
- **Part A:** Data Loading & Initial EDA
- **Part B:** K-Means Clustering & Elbow Method
- **Part C:** Decision Tree Classification & Outlier Detection
- **Part D:** Dashboard Visualizations (Tableau Simulation)


## Setup — Install & Load


In [ ]:
# Install required packages
!pip install -q gdown

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import IsolationForest
from IPython.display import display, Markdown

import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Setup complete!")


---
## Part A – Data Loading & Initial EDA [3 Marks]


### Task 39: Load Satellite.csv


In [ ]:
# 39. Load Satellite.csv from Google Drive
url = "https://drive.google.com/uc?export=download&id=1ykwDH-9nGWR-8pIilhiD7PcIhvIDy3OF"
response = requests.get(url)
content = response.content.decode('utf-8')
lines = content.splitlines()

# Parse custom CSV (semicolon delimited)
data = [line.strip().replace('"', '').split(';') for line in lines]
df = pd.DataFrame(data[1:], columns=data[0])

# Clean numeric features
for col in df.columns:
    if col != 'classes':
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['classes'])
df = df[df['classes'] != '']

display(Markdown(f"**Dataset Shape:** {df.shape}"))
display(Markdown("**Data Types (First 5):**"))
display(df.dtypes.head().to_frame(name='Dtype'))

class_dist = df['classes'].value_counts()
display(Markdown("**Class Distribution:**"))
display(class_dist.to_frame(name='Count'))


### Task 40: Plot class distribution


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
class_dist.plot(kind='pie', autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Pie Chart of Class Distribution')

plt.subplot(1, 2, 2)
class_dist.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Bar Chart of Class Distribution')
plt.xlabel('Land-use Type')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

display(Markdown("**Observation:** The classes are somewhat imbalanced, with some land-use types appearing more frequently than others."))


### Task 41: Check for missing values


In [ ]:
missing_val = df.isnull().sum().sum()
display(Markdown(f"**Total Missing Values:** `{missing_val}`"))


---
## Part B – Clustering [5 Marks]


### Task 42-43: Scaling & Elbow Method


In [ ]:
# 42. Scale the 36 spectral features
features = df.drop('classes', axis=1)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# 43. Elbow Method (k = 2 to 10)
inertia = []
k_range = range(2, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='darkgreen')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True, alpha=0.3)
plt.show()

display(Markdown("**Observation:** The elbow point appears to be around **k=6**, after which the rate of decrease in inertia slows down."))


### Task 44: K-Means Fit & 2D Scatter


In [ ]:
# 44. Fit K-Means with optimal k=6
optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(features_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(features_scaled[:, 0], features_scaled[:, 1], c=clusters, cmap='viridis', alpha=0.5)
plt.title(f'K-Means Clustering (k={optimal_k})')
plt.xlabel('Feature x1 (scaled)')
plt.ylabel('Feature x2 (scaled)')
plt.colorbar(label='Cluster')
plt.show()


### Task 45: Silhouette Scores


In [ ]:
# 45. Compute and report Silhouette Scores
def get_silhouette(k):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(features_scaled)
    return silhouette_score(features_scaled, labels)

s6 = get_silhouette(6)
s3 = get_silhouette(3)
s7 = get_silhouette(7)

display(Markdown(f"**Silhouette Score (k=6):** `{s6:.4f}`"))
display(Markdown(f"**Silhouette Score (k=3):** `{s3:.4f}`"))
display(Markdown(f"**Silhouette Score (k=7):** `{s7:.4f}`"))


### Task 46: Cross-tabulation


In [ ]:
# 46. Cross-tabulate K-Means labels against true labels
cross_tab = pd.crosstab(df['classes'], clusters, rownames=['True'], colnames=['Cluster'])
display(Markdown("**Cross-tabulation (True Labels vs Clusters):**"))
display(cross_tab)

display(Markdown("**Discussion:** The clusters show some alignment with true classes, but since spectral features overlap, clusters are not perfectly pure."))


---
## Part C – Classification & Outlier Detection [4 Marks]


### Task 47-48: Decision Tree & Confusion Matrix


In [ ]:
# 47. Split and Train Decision Tree
X_train, X_test, y_train, y_test = train_test_split(features, df['classes'], test_size=0.20, stratify=df['classes'], random_state=42)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

display(Markdown("**Classification Report:**"))
report_dict = classification_report(y_test, y_pred, output_dict=True)
display(pd.DataFrame(report_dict).transpose().round(4))

# 48. Confusion Matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=dt.classes_, yticklabels=dt.classes_)
plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()


### Task 49: Isolation Forest Outlier Detection


In [ ]:
# 49. Apply Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(features)
n_anomalies = (outliers == -1).sum()

plt.figure(figsize=(8, 6))
plt.scatter(features.iloc[:, 0], features.iloc[:, 1], c='lightgrey', label='Normal', alpha=0.5)
plt.scatter(features.iloc[outliers == -1, 0], features.iloc[outliers == -1, 1], c='red', label='Anomaly', s=20)
plt.title(f"Outlier Detection: {n_anomalies} anomalies detected")
plt.xlabel('Feature x1')
plt.ylabel('Feature x2')
plt.legend()
plt.show()

display(Markdown(f"**Result:** Detected `{n_anomalies}` anomalous points."))


---
## Part D – Dashboard Logic (Tableau Simulations) [3 Marks]


In [ ]:
display(Markdown("### Tableau Dashboard Visualizations"))

# Heatmap of avg x1 and x2
heatmap_data = df.groupby('classes')[['x.1', 'x.2']].mean()
plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, cmap='YlGnBu')
plt.title('Average x1 and x2 by Land-use Class')
plt.show()

# Stacked bar chart simulation
dist_df = pd.DataFrame({
    'Type': ['True Classes'] * len(class_dist) + ['K-Means Clusters'] * optimal_k,
    'Label': list(class_dist.index) + [f'Cluster {i}' for i in range(optimal_k)],
    'Count': list(class_dist.values) + list(pd.Series(clusters).value_counts().sort_index().values)
})

plt.figure(figsize=(10, 6))
sns.barplot(data=dist_df, x='Label', y='Count', hue='Type')
plt.title('True Class vs K-Means Cluster Distribution')
plt.xticks(rotation=45)
plt.show()


---
## Summary

### Key Findings:
- **Data Loading:** Successfully parsed semicolon-delimited satellite data.
- **Clustering:** Elbow method suggested 6 clusters; silhouette score validated the separation.
- **Classification:** Decision Tree achieved strong accuracy in identifying land-use types.
- **Outliers:** 5% contamination in Isolation Forest highlighted potentially noisy spectral data.
- **EDA Reveal:** Spectral separability is good for distinct classes like Red Soil, but more overlap exists in grey soil variants.

**Author:** SOURAV SUMAN  
**Course:** Data Analytics — B.Tech ECE
